# 项目 · 连锁零售经营分析（任务书）

> 这是 SQL 模块的收官实战：把 00~06 课学到的 SELECT / 聚合 / JOIN / 子查询 / 窗口函数 / CTE，用到一个完整的业务分析里。
> 目标不是“把 SQL 写对”，而是像真实数据分析师那样：**从数据里发现问题 → 用数字量化 → 给出业务建议**。
>
> **请先独立完成全部任务，不要急着看参考答案。卡住了再翻 `SQL零售经营分析项目_参考答案.ipynb`。**

---

## 一、业务背景

你是一家连锁零售公司的数据分析师。公司有 5 个销售大区、300 个客户、60 个商品、2600 张订单（2022-01 ~ 2025-12）。CEO 要一份年度经营复盘，想知道：

1. 整体盘子有多大、赚不赚钱；
2. 钱是从哪类商品、哪个地区赚来的；
3. 客户有没有复购、哪些是高价值客户；
4. 组织/人员结构健不健康；
5. 增长是在加速还是失速；
6. 有没有“看着热闹其实在亏钱”的异常。

数据在 `data/shop.db`（规范化多表库）。

## 二、数据字典

| 表 | 行数 | 关键列 | 说明 |
|---|---|---|---|
| customers | 300 | customer_id PK, name, gender, age, segment, region, city, signup_date | 客户主数据 |
| products | 60 | product_id PK, product_name, category, subcategory, unit_price, unit_cost | 商品目录 |
| employees | 54 | employee_id PK, name, department, title, region, hire_date, salary, manager_id→employees | 员工（自关联层级） |
| orders | 2600 | order_id PK, customer_id FK, order_date, ship_date, ship_mode, region, city | 订单主表 |
| order_items | 6407 | item_id PK, order_id FK, product_id FK, quantity, discount, sales, profit | 订单明细（一对多） |

记住 03 课的教训：orders 和 order_items 是一对多，连起来行数会膨胀；数独立订单/客户用 `COUNT(DISTINCT 主键)`。

## 三、分模块任务

### 模块 1 · 销售总览（KPI 大盘）
1.1 算出：总销售额、总利润、订单总数、下单客户数、平均每单销售额（客单价=总销售额/订单数）。
1.2 算整体利润率 = 总利润 / 总销售额。
1.3 思考：总销售额和总利润分别来自哪张表？为什么客单价要用“订单数”而不是“明细行数”？

### 模块 2 · 品类与地区结构
2.1 按 `category` 汇总销售额、利润、利润率，找出“最赚钱的大类”和“利润率最低的大类”。
2.2 按 `region` 汇总销售额与利润占比。
2.3 用窗口函数给各大类销售额做“占全公司百分比”。

### 模块 3 · 客户复购与 RFM
3.1 以数据中最新订单日期为“今天”，给每个客户算 R（距今天数）、F（下单次数）、M（累计消费额）。
3.2 用 `NTILE(4)` 按 M 把客户分 4 档，统计每档客户数。
3.3 找出“最近 90 天没有下单”的沉睡客户数。
3.4 思考：RFM 里哪个指标最该优先运营？为什么？

### 模块 4 · 员工与组织分析
4.1 在每个 `department` 内部，按薪资 `salary` 降序排名（用 RANK），列出销售部全员薪资排名。
4.2 用递归 CTE 从大老板往下，打印组织层级 `level` 和汇报路径 `path`。
4.3 统计每个部门人数、平均薪资。

### 模块 5 · 累计与同比环比
5.1 按年汇总销售额，用 `LAG` 算每年同比增长率。
5.2 按年做累计销售额（从 2022 年累加到当年）。
5.3 按月做近 12 个月的销售额，用 LAG 算环比（自己挑最近一年）。

### 模块 6 · 组内 TopN 与异常发现
6.1 每个 `region` 销售额 Top2 的城市。
6.2 找出 `profit < 0` 的订单明细有多少行、合计亏了多少。
6.3 按 `discount` 分组，看哪一档折扣在亏钱——这是一个“促销异常”。
6.4 （开放）自己再找一个你认为值得关注的异常或规律。

## 四、交付规范

- 每个模块至少 1 条 SQL，尽量用 CTE（WITH）保持可读；
- 每个模块结尾写 1~3 行 Markdown 结论，格式为 **“发现 + 数字 + 建议”**，例如：
  > 发现：办公用品销售额仅占 1.6%，远低于家具/科技。数字：39.6 万 vs 1725 万。建议：评估是否精简 SKU 或捆绑销售。
- 最终在 notebook 末尾汇总一份“给 CEO 的 5 条结论”。
- 提交物：本任务书 + 你自己的解答 notebook，可上传 GitHub 作为作品集。

## 五、检查点（自检清单）

- [ ] 所有 SQL 能在 `shop.db` 上跑通，没有“列不存在”错误
- [ ] 涉及一对多连接时，数订单/客户都用了 COUNT DISTINCT
- [ ] 组内 TopN 用了窗口函数，而不是手写相关子查询
- [ ] RFM 的 R 用了日期函数（julianday / strftime）
- [ ] 每个模块都有“发现+数字+建议”

In [1]:
# 先连库。参考 00 课 BOOT 样板，下面是给你准备好的起点：
import sqlite3
import pandas as pd

SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
pd.set_option("display.width", 220)
conn = sqlite3.connect(SHOP_DB)
def q(sql):
    return pd.read_sql_query(sql, conn)

# ===== 在下面每个模块对应的 TODO 单元格里写你的 SQL =====

### 模块 1 · 销售总览

In [2]:
# TODO 1.1~1.3：总销售额/总利润/订单数/客户数/客单价/利润率

In [3]:
# TODO 1.3：用一句 Markdown 写出你的大盘结论（发现+数字+建议）

### 模块 2 · 品类与地区结构

In [4]:
# TODO 2.1：按 category 汇总销售额/利润/利润率

In [5]:
# TODO 2.2：按 region 汇总销售额/利润

In [6]:
# TODO 2.3：窗口函数算各大类占比

In [7]:
# TODO 2.3：Markdown 结论

### 模块 3 · 客户复购与 RFM

In [8]:
# TODO 3.1：每个客户的 R/F/M

In [9]:
# TODO 3.2：NTILE 按 M 分 4 档

In [10]:
# TODO 3.3：近 90 天未下单的沉睡客户数

In [11]:
# TODO 3.4：Markdown 结论

### 模块 4 · 员工与组织分析

In [12]:
# TODO 4.1：部门内薪资 RANK

In [13]:
# TODO 4.2：递归 CTE 打印组织层级

In [14]:
# TODO 4.3：各部门人数与平均薪资

### 模块 5 · 累计与同比环比

In [15]:
# TODO 5.1：年度销售额 + LAG 同比

In [16]:
# TODO 5.2：年度累计销售额

In [17]:
# TODO 5.3：月度环比

### 模块 6 · 组内 TopN 与异常发现

In [18]:
# TODO 6.1：每地区 Top2 城市

In [19]:
# TODO 6.2：负利润明细行数与金额

In [20]:
# TODO 6.3：按 discount 分组找亏钱折扣档

In [21]:
# TODO 6.4：自己再找一个异常

### 给 CEO 的 5 条结论（汇总）

In [22]:
# TODO：在下面用 Markdown 或 print 汇总你的 5 条发现
# 格式：发现 + 数字 + 建议

## 来源与延伸

- 本项目数据由本模块 `data/build_databases.py` 生成（种子 20260913，可复现）。
- 进阶可把同样的分析迁移到 `data/superstore.db`（真实 Superstore 数据，单表 orders，9994 行）做对比练习。
- 真实世界练手库：Chinook https://github.com/lerocha/chinook-database ；Northwind-SQLite3 https://github.com/jpwhite3/northwind-SQLite3
- 面试复盘可配合 DataLemur https://datalemur.com/ 与 LeetCode Database https://leetcode.com/problemset/database/